In [1]:
import os
os.chdir("/workspaces/pythonProject/medallion-turso-v3/notebooks")

# 01 · Load the mock source data into Turso (sales domain)
Creates the four source tables of the **sales** domain in Turso (`customers`, `products`, `sales`, `returns`) and loads one CSV batch (upsert, so batches can be re-applied safely). The table list and keys come from `domains/sales.yml`.

* **batch 1**: initial load (60 customers, 20 products, 310 sales lines, 30 returns, with deliberate bad data and **duplicate order lines**: the same line submitted again under a new id)
* **batch 2**: changes, new rows, sales and returns for customers that do not exist yet, and two more duplicates of lines already loaded in batch 1
* **batch 3**: the missing customers arrive, and one sale line is corrected to a different customer (the return that refers to that line follows it in `fact_returns`)

Set the Codespaces secrets `TURSO_DATABASE_URL` and `TURSO_AUTH_TOKEN` first (the kernel must be the Codespace kernel so it sees them).

In [2]:
import os
BATCH = int(os.getenv("BATCH", 1))              # 1, 2 or 3: edit here, then run the pipeline notebook after each batch
RESET = os.getenv("RESET_SOURCE") == "1"        # True = start over: drops this domain's source, Bronze, Silver and Gold tables

In [3]:
%run -i ./00_config.ipynb

rule library: 27 checks for 9 entities


In [4]:
open_warehouse()
print("connected to Turso")

SALES = load_domain("sales")                                           # tables and keys come from domains/sales.yml
KEYS = {e: ENTITIES[e]["keys"][0] for e in SALES["entities"]}

if RESET: reset_domain(SALES)

ddl = "\n".join(l for l in find_file("schema.sql").read_text().splitlines() if not l.strip().startswith("--"))
for stmt in [s.strip() for s in ddl.split(";") if s.strip()]:
    con.execute(stmt)
con.commit()

connected to Turso
domain 'sales': 4 entities, 2 dimensions, 2 facts


In [5]:
for table in KEYS:
    try:
        f = find_file(f"{table}_batch{BATCH}.csv")
    except FileNotFoundError:
        continue                                  # not every batch has a file for every table
    n, secs = upsert_csv(table, f, KEYS[table])
    print(f"{table:10s} loaded {n:4d} rows from {f.name} in {secs:.1f}s")

customers  loaded   60 rows from customers_batch1.csv in 0.1s
products   loaded   20 rows from products_batch1.csv in 0.1s
sales      loaded  310 rows from sales_batch1.csv in 0.1s
returns    loaded   30 rows from returns_batch1.csv in 0.1s


In [6]:
for t in KEYS:
    print(t, con.one(f"SELECT count(*), min(updated_at), max(updated_at) FROM {t}"))

customers (60, '2026-09-02 08:45:40', '2026-10-30 14:13:55')
products (20, '2026-09-05 19:17:22', '2026-10-31 18:47:42')
sales (310, '2026-09-01 09:56:16', '2026-10-31 20:05:30')
returns (30, '2026-09-06 15:41:04', '2026-10-28 14:00:00')


In [7]:
release_all()          # closes the Turso connection (the next notebook opens its own)